# Gradient Theory Analysis（论文探索版）

本Notebook按你要求重构，覆盖以下探索维度：

1. **分组统计**：`step / adapter_type / layer_depth / 全局 / grad_partition(text_only, all)`
2. **梯度信息**：范数、统计量、相似度（cos）
3. **容量分析**：按 `grad_partition` 的 SVD 奇异值谱（Singular Value Spectrum）
4. **频谱分析**：按 `grad_partition` 的梯度频谱图
5. **模态空间分析**：按 `layer_depth` 的 text/image hidden state PCA / t-SNE
6. **ratio重加权分析**：由 `text_token_ratio / image_token_ratio` 将 `all` 梯度分解为 text/image proxy，再按 step 平均并可视化


In [ ]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 200)


In [ ]:
GRAD_LOG = Path('gradient_logs.jsonl')
HS_LOG = Path('hidden_state_logs.jsonl')
OUT = Path('analysis_outputs')
OUT.mkdir(parents=True, exist_ok=True)

STEP_MIN = None
STEP_MAX = None
STEP_MOD = None
MAX_ROWS = 300000
MAX_VEC_ROWS = 120000
MAX_TSNE_SAMPLES = 4000

assert GRAD_LOG.exists(), f'Missing {GRAD_LOG}'
print('gradient log:', GRAD_LOG.resolve())
print('hidden-state log exists:', HS_LOG.exists())


In [ ]:
def _step_ok(v, lo=None, hi=None, mod=None):
    try:
        s = int(v)
    except Exception:
        return False
    if lo is not None and s < lo:
        return False
    if hi is not None and s > hi:
        return False
    if mod is not None and mod > 1 and s % mod != 0:
        return False
    return True


def load_jsonl(path: Path, usecols=None, row_filter=None, max_rows=None):
    usecols = set(usecols) if usecols else None
    rows = []
    if not path.exists():
        return pd.DataFrame()

    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            if 'step' in obj and not _step_ok(obj['step'], STEP_MIN, STEP_MAX, STEP_MOD):
                continue
            if row_filter is not None and not row_filter(obj):
                continue

            rows.append(obj if usecols is None else {k: obj.get(k, None) for k in usecols})
            if max_rows and len(rows) >= max_rows:
                break

    return pd.DataFrame(rows)


def parse_vector_field(x):
    if x is None:
        return None
    if isinstance(x, (list, tuple, np.ndarray)):
        return np.asarray(x, dtype=np.float32).reshape(-1)
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return None
        p = Path(s)
        if s.endswith('.npy') and p.exists():
            try:
                return np.load(p).reshape(-1)
            except Exception:
                return None
        for parser in (json.loads, ast.literal_eval):
            try:
                y = parser(s)
                if isinstance(y, (list, tuple)):
                    return np.asarray(y, dtype=np.float32).reshape(-1)
            except Exception:
                pass
    return None


## 1) 数据加载与基础清洗

In [ ]:
gdf = load_jsonl(
    GRAD_LOG,
    usecols=[
        'step','adapter_type','layer_depth','grad_partition','param_name',
        'grad_norm','grad_mean','grad_std','grad_abs_mean',
        'text_token_ratio','image_token_ratio',
        'grad_path','grad','grad_was_none'
    ],
    max_rows=MAX_ROWS,
)

if len(gdf) == 0:
    raise ValueError('No gradient rows loaded')

for c in ['step','layer_depth','grad_norm','grad_mean','grad_std','grad_abs_mean','text_token_ratio','image_token_ratio']:
    if c in gdf.columns:
        gdf[c] = pd.to_numeric(gdf[c], errors='coerce')
if 'grad_was_none' in gdf.columns:
    gdf['grad_was_none'] = gdf['grad_was_none'].fillna(False).astype(bool)
if 'grad_partition' in gdf.columns:
    gdf['grad_partition'] = gdf['grad_partition'].fillna('all')

gdf = gdf[gdf['grad_partition'].isin(['all','text_only'])].copy()

print('rows:', len(gdf))
print('partitions:', sorted(gdf['grad_partition'].dropna().unique().tolist()))
print('adapter types:', gdf['adapter_type'].dropna().nunique())


## 2) 分组探索：step / adapter_type / layer_depth / 全局 / grad_partition

In [ ]:
global_stats = gdf.groupby('grad_partition', as_index=False).agg(
    grad_norm_mean=('grad_norm','mean'),
    grad_norm_std=('grad_norm','std'),
    grad_mean_mean=('grad_mean','mean'),
    grad_std_mean=('grad_std','mean'),
    grad_abs_mean_mean=('grad_abs_mean','mean'),
    none_ratio=('grad_was_none','mean')
)

display(global_stats)
global_stats.to_csv(OUT/'table_global_by_partition.csv', index=False, encoding='utf-8-sig')

step_stats = gdf.groupby(['step','grad_partition'], as_index=False)['grad_norm'].mean()
plt.figure(figsize=(12,5))
sns.lineplot(data=step_stats, x='step', y='grad_norm', hue='grad_partition')
plt.title('Step-level mean grad_norm by grad_partition')
plt.tight_layout(); plt.savefig(OUT/'fig_step_grad_norm_partition.png', dpi=180); plt.show()

layer_stats = gdf.groupby(['layer_depth','grad_partition'], as_index=False)['grad_norm'].mean()
plt.figure(figsize=(11,5))
sns.lineplot(data=layer_stats, x='layer_depth', y='grad_norm', hue='grad_partition', marker='o')
plt.title('Layer-depth mean grad_norm by grad_partition')
plt.tight_layout(); plt.savefig(OUT/'fig_layer_grad_norm_partition.png', dpi=180); plt.show()

a_l_stats = gdf.groupby(['adapter_type','layer_depth','grad_partition'], as_index=False)['grad_norm'].mean()
a_l_stats.to_csv(OUT/'table_adapter_layer_partition_grad_norm.csv', index=False, encoding='utf-8-sig')


## 3) 相似度分析（cos）：按 step / adapter_type / layer_depth

In [ ]:
vdf = gdf[['step','adapter_type','layer_depth','grad_partition','param_name','grad_path','grad']].copy()

vdf['vec'] = vdf['grad_path'].apply(parse_vector_field)
miss = vdf['vec'].isna()
if miss.any():
    vdf.loc[miss, 'vec'] = vdf.loc[miss, 'grad'].apply(parse_vector_field)

vdf = vdf[vdf['vec'].notna()].copy()
print('vector rows:', len(vdf))

if len(vdf) > 0:
    t = vdf[vdf['grad_partition']=='text_only'][['step','adapter_type','layer_depth','param_name','vec']].rename(columns={'vec':'tvec'})
    a = vdf[vdf['grad_partition']=='all'][['step','adapter_type','layer_depth','param_name','vec']].rename(columns={'vec':'avec'})
    pair = t.merge(a, on=['step','adapter_type','layer_depth','param_name'], how='inner')

    def cosine(x, y, eps=1e-12):
        m = min(len(x), len(y))
        if m == 0:
            return np.nan
        x = x[:m]; y = y[:m]
        return float(np.dot(x, y) / (np.linalg.norm(x)*np.linalg.norm(y) + eps))

    pair['cos_ta'] = pair.apply(lambda r: cosine(r['tvec'], r['avec']), axis=1)
    pair = pair.dropna(subset=['cos_ta'])

    display(pair['cos_ta'].describe())

    cos_step = pair.groupby('step', as_index=False)['cos_ta'].mean()
    plt.figure(figsize=(12,4))
    sns.lineplot(data=cos_step, x='step', y='cos_ta')
    plt.axhline(0, color='r', linestyle='--')
    plt.title('Step-level cosine(text_only, all)')
    plt.tight_layout(); plt.savefig(OUT/'fig_step_cos_text_all.png', dpi=180); plt.show()

    cos_layer = pair.groupby('layer_depth', as_index=False)['cos_ta'].mean()
    plt.figure(figsize=(11,4))
    sns.lineplot(data=cos_layer, x='layer_depth', y='cos_ta', marker='o')
    plt.axhline(0, color='r', linestyle='--')
    plt.title('Layer-depth cosine(text_only, all)')
    plt.tight_layout(); plt.savefig(OUT/'fig_layer_cos_text_all.png', dpi=180); plt.show()

    cos_al = pair.groupby(['adapter_type','layer_depth'], as_index=False)['cos_ta'].mean()
    cos_al.to_csv(OUT/'table_adapter_layer_cos_text_all.csv', index=False, encoding='utf-8-sig')


## 4) 按 grad_partition 的 SVD 奇异值谱（容量可视化）

In [ ]:
if len(vdf) > 0:
    spec_rows = []
    for part, g in vdf.groupby('grad_partition'):
        vecs = g['vec'].tolist()
        if len(vecs) < 2:
            continue
        dmin = min(len(v) for v in vecs)
        if dmin <= 1:
            continue
        X = np.stack([v[:dmin] for v in vecs], axis=0)
        X = X - X.mean(axis=0, keepdims=True)
        s = np.linalg.svd(X, full_matrices=False, compute_uv=False)
        sv = s / (s.sum() + 1e-12)
        for i, val in enumerate(sv[:128], 1):
            spec_rows.append({'grad_partition': part, 'rank_idx': i, 'singular_value_norm': float(val)})

    spec_df = pd.DataFrame(spec_rows)
    if len(spec_df) > 0:
        spec_df.to_csv(OUT/'table_singular_spectrum_by_partition.csv', index=False, encoding='utf-8-sig')
        plt.figure(figsize=(11,5))
        sns.lineplot(data=spec_df, x='rank_idx', y='singular_value_norm', hue='grad_partition')
        plt.title('SVD Singular Value Spectrum by grad_partition')
        plt.tight_layout(); plt.savefig(OUT/'fig_svd_spectrum_partition.png', dpi=180); plt.show()


## 5) 按 grad_partition 的梯度频谱图（FFT）

In [ ]:
if len(vdf) > 0:
    fs_rows = []
    for part, g in vdf.groupby('grad_partition'):
        vecs = g['vec'].tolist()
        if len(vecs) == 0:
            continue
        dmin = min(len(v) for v in vecs)
        if dmin <= 4:
            continue
        X = np.stack([v[:dmin] for v in vecs], axis=0)
        amp = np.abs(np.fft.rfft(X, axis=1))
        mean_amp = amp.mean(axis=0)
        freq = np.arange(mean_amp.shape[0])
        for fidx, aval in zip(freq, mean_amp):
            fs_rows.append({'grad_partition': part, 'freq_idx': int(fidx), 'amplitude': float(aval)})

    fs_df = pd.DataFrame(fs_rows)
    if len(fs_df) > 0:
        fs_df.to_csv(OUT/'table_gradient_spectrum_by_partition.csv', index=False, encoding='utf-8-sig')
        plt.figure(figsize=(11,5))
        sns.lineplot(data=fs_df, x='freq_idx', y='amplitude', hue='grad_partition')
        plt.title('Gradient Frequency Spectrum by grad_partition')
        plt.tight_layout(); plt.savefig(OUT/'fig_gradient_spectrum_partition.png', dpi=180); plt.show()


## 6) token_ratio 重加权：从 all 构建 text/image proxy 梯度并做 step-level 平均

In [ ]:
all_df = gdf[gdf['grad_partition']=='all'].copy()
if len(all_df) > 0:
    all_df['text_token_ratio'] = all_df['text_token_ratio'].fillna(0.0)
    all_df['image_token_ratio'] = all_df['image_token_ratio'].fillna(0.0)

    all_df['grad_norm_text_proxy'] = all_df['grad_norm'] * all_df['text_token_ratio']
    all_df['grad_norm_image_proxy'] = all_df['grad_norm'] * all_df['image_token_ratio']

    step_proxy = all_df.groupby('step', as_index=False).agg(
        text_proxy_mean=('grad_norm_text_proxy','mean'),
        image_proxy_mean=('grad_norm_image_proxy','mean'),
    )

    step_proxy['proxy_gap'] = step_proxy['image_proxy_mean'] - step_proxy['text_proxy_mean']
    step_proxy.to_csv(OUT/'table_step_proxy_gradients.csv', index=False, encoding='utf-8-sig')

    plt.figure(figsize=(12,5))
    plt.plot(step_proxy['step'], step_proxy['text_proxy_mean'], label='text_proxy')
    plt.plot(step_proxy['step'], step_proxy['image_proxy_mean'], label='image_proxy')
    plt.title('Step-level proxy gradients from token ratios')
    plt.legend(); plt.tight_layout(); plt.savefig(OUT/'fig_step_proxy_gradients.png', dpi=180); plt.show()

    plt.figure(figsize=(12,4))
    plt.plot(step_proxy['step'], step_proxy['proxy_gap'])
    plt.axhline(0, color='r', linestyle='--')
    plt.title('Step-level proxy gap (image - text)')
    plt.tight_layout(); plt.savefig(OUT/'fig_step_proxy_gap.png', dpi=180); plt.show()


## 7) layer-depth 的 text/image hidden-state 空间可视化（PCA / t-SNE）

In [ ]:
hdf = load_jsonl(
    HS_LOG,
    usecols=['step','layer_depth','modality','hidden_path','hidden_norm','sample_index'],
    max_rows=MAX_ROWS,
)

if len(hdf) == 0:
    print('No hidden_state_logs.jsonl detected; skip hidden-space section.')
else:
    for c in ['step','layer_depth','hidden_norm','sample_index']:
        if c in hdf.columns:
            hdf[c] = pd.to_numeric(hdf[c], errors='coerce')

    layer_norm = hdf.groupby(['layer_depth','modality'], as_index=False)['hidden_norm'].mean()
    plt.figure(figsize=(11,5))
    sns.lineplot(data=layer_norm, x='layer_depth', y='hidden_norm', hue='modality', marker='o')
    plt.title('Layer-depth hidden norm by modality')
    plt.tight_layout(); plt.savefig(OUT/'fig_layer_hidden_norm_modality.png', dpi=180); plt.show()

    layers = sorted(hdf['layer_depth'].dropna().unique().tolist())
    pick = []
    if layers:
        pick.append(layers[0])
    if len(layers) > 2:
        pick.append(layers[len(layers)//2])
    if len(layers) > 1:
        pick.append(layers[-1])
    pick = sorted(list(set(pick)))
    print('selected layers:', pick)

    for layer in pick:
        sub = hdf[hdf['layer_depth']==layer].copy()
        vecs, labs = [], []
        for _, r in sub.iterrows():
            v = parse_vector_field(r.get('hidden_path'))
            if v is None:
                continue
            vecs.append(v)
            labs.append(r.get('modality', 'unknown'))

        if len(vecs) < 20:
            continue

        X = np.stack(vecs, axis=0)
        y = np.array(labs)

        Zp = PCA(n_components=2, random_state=42).fit_transform(X)
        pca_df = pd.DataFrame({'x': Zp[:,0], 'y': Zp[:,1], 'modality': y})
        plt.figure(figsize=(6,5))
        sns.scatterplot(data=pca_df, x='x', y='y', hue='modality', s=12, alpha=0.7)
        plt.title(f'Layer {int(layer)} PCA hidden-space')
        plt.tight_layout(); plt.savefig(OUT/f'fig_layer{int(layer)}_pca_hidden.png', dpi=180); plt.show()

        Xts, yts = X, y
        if X.shape[0] > MAX_TSNE_SAMPLES:
            idx = np.random.choice(X.shape[0], MAX_TSNE_SAMPLES, replace=False)
            Xts, yts = X[idx], y[idx]
        if Xts.shape[0] >= 50:
            Zt = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto').fit_transform(Xts)
            ts_df = pd.DataFrame({'x': Zt[:,0], 'y': Zt[:,1], 'modality': yts})
            plt.figure(figsize=(6,5))
            sns.scatterplot(data=ts_df, x='x', y='y', hue='modality', s=12, alpha=0.7)
            plt.title(f'Layer {int(layer)} t-SNE hidden-space')
            plt.tight_layout(); plt.savefig(OUT/f'fig_layer{int(layer)}_tsne_hidden.png', dpi=180); plt.show()


## 8) 适合论文的可视化建议（基于本Notebook可直接产出）

推荐至少放入论文的图：
1. **Step-level grad_norm 曲线（all vs text_only）**：展示整体优化动态。
2. **Layer-depth cosine(text_only, all) 曲线**：展示层级冲突/一致性。
3. **SVD 奇异值谱对比图（by partition）**：展示容量差异。
4. **梯度频谱图（by partition）**：展示更新信号频率结构差异。
5. **Layer-depth hidden-space PCA/t-SNE（前/中/后层）**：展示模态表征分离/融合轨迹。
6. **Step-level proxy gap（image-text）**：展示 token ratio 驱动下的权重转移趋势。
